# Family-Ladder Scaling Deduction Study

This is the DEDUCTION side of the same family-ladder scaling study
`notebooks/induction/induction_eval.ipynb` documents. The induction side
holds a fixed quiz and measures accuracy on periodic-sequence completion
across 21 checkpoints; this side holds a fixed Lean 4 theorem-proving sweep
and measures **next-tactic success** across the SAME 21 checkpoints -- 7
vendor families x 3 rungs each (smallest / geometric-middle / largest
checkpoint on that family's public ladder). The sweep configuration --
seed, decoding params, theorem pool selector, and the four prompt rungs
below -- lives in `notebooks/deduction/run_study.build_config`, and is
IMPORTED here rather than re-declared, for the same reason the induction
notebook imports rather than re-declares its own config: a hand-copied
sweep here would drift from the sweep the fleet actually runs the moment
either one is edited.

## What varies, what's held fixed

The theorem pool and prompt-rendering rungs are held FIXED across every
checkpoint; what varies is the MODEL -- the identical 21-checkpoint roster
(spec keys, analysis tags, instance tiers) the induction notebook documents
in its own roster table, imported from the same `run_study.MODELS` /
`run_study.COT_ARGS` tables this file re-exports (see `notebooks/deduction/
run_study.py`'s module docstring, "WHAT THIS IS"). Holding the sweep fixed
is what makes a next-tactic success-rate difference between two checkpoints
attributable to parameter count rather than to a changed sweep.

## The four rungs

`build_config`'s `"rungs"` key runs exactly four context rungs per theorem,
out of the larger `stepk:0..2` / `hint:0..4` ladder `context.py` implements
(see `smolbench.deduction.lean.context`'s module docstring for the full
ladder and how it is meant to be used):

- **`stepk:1`** -- the current goal plus the full tactic state (hypotheses
  and goal), no answer-conditional content. The baseline rung.
- **`hint:2`** -- `stepk:2` (adds the proof-so-far and theorem identity)
  plus the full source and proof of every premise the TRUE next tactic
  uses. Substantial answer-conditional information.
- **`noise:3`** -- a TOKEN-MATCHED LENGTH CONTROL for `hint:3` (below): the
  exact `hint:2` baseline, padded with whitespace to `hint:3`'s exact
  token count. Comparing `hint:3` against `noise:3` isolates the effect of
  `hint:3`'s extra CONTENT, because prompt length is no longer a
  confound -- both rungs are the same number of tokens, and the padding
  carries no information a model could use (see
  `smolbench.deduction.lean.context._render_noise_parts`'s docstring for
  the exact-match mechanics).
- **`hint:3`** -- `hint:2` plus a 1-hop transitive closure over those
  premises' own dependencies: more answer-conditional content, at a real
  token cost.

## Replicates and pool

`n_replicates = 1` (`R=1`) over 300 theorems drawn from the
`replay_passing`/`novel_premises`/`val` pool -- see `notebooks/deduction/
README.md` for the pool's real size and how the sidecar it is drawn from
was built.

## Cost warning

Exactly as the induction notebook: this study provisions up to 21
concurrent EC2 spot instances, and **nothing in this notebook can
provision or bill anything.** The deduction phase is launched from a
**terminal**, outside any kernel session (see "Fleet Launch" below), and
normally REATTACHES to the box its own induction phase already provisioned
rather than launching a second one per lane. Running this notebook
top-to-bottom does **not** launch, and cannot accidentally launch, any of
the 21 boxes.


In [ ]:
import logging
import sys
from pathlib import Path

from dotenv import load_dotenv

# CRITICAL, import-order trap: smolbench.evals.ec2 freezes its EC2_* module
# constants (EC2_EXPERIMENT_TAG, EC2_INSTANCE_TYPES, ...) from os.environ at
# IMPORT time, not at call time (see that module's docstring, "Env-read
# timing"). load_dotenv MUST therefore run before the first `smolbench`
# import anywhere in this kernel, or this study's config is silently frozen
# to un-overridden defaults for the life of the kernel -- with no error to
# signal it.

# Resolve the notebook's own directory without relying on cwd. The sibling
# periodic_moe notebook's cell 1 does `load_dotenv(Path.cwd() / "keys.env")`
# unguarded -- a kernel started from the repo root, or from any other cwd,
# makes that silently load nothing (load_dotenv returns quietly on a missing
# path) and every EC2_* constant below freezes to its default. We do NOT
# copy that pattern: walk up from cwd looking for the repo root instead.
NB_DIR = Path.cwd() if (Path.cwd() / "keys.env").exists() else None
if NB_DIR is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "smolbench").is_dir():
            NB_DIR = candidate / "notebooks" / "deduction"
            break

assert NB_DIR is not None and (NB_DIR / "keys.env").exists(), (
    "could not locate notebooks/deduction/keys.env by walking up from this "
    f"kernel's cwd ({Path.cwd()}) to a directory containing both "
    "pyproject.toml and smolbench/. Start the kernel from inside the repo "
    "(or a subdirectory of it)."
)
# REPO_ROOT anchors the `scripts` namespace-package import in the fleet
# status section below -- it is NB_DIR's grandparent (notebooks/deduction ->
# notebooks -> repo root).
REPO_ROOT = NB_DIR.parent.parent

load_dotenv(NB_DIR / "keys.env", verbose=True)
logging.basicConfig(level=logging.INFO)

# Unlike notebooks/induction/run_study.py, THIS study's run_study.py
# (imported below) does NOT call load_dotenv on this same keys.env file --
# see its module docstring's "MODULE IMPORT ORDER" section: it only ever
# triggers load_dotenv indirectly, on notebooks/induction/run_study.py's OWN
# keys.env, when it loads that file by path to import MODELS/COT_ARGS. So
# this cell's load_dotenv call on notebooks/deduction/keys.env is the ONLY
# thing in this kernel that ever reads that file -- load-bearing, not
# belt-and-braces, for any variable only notebooks/deduction/keys.env sets
# (e.g. ANALYSIS_HOST, or an AWS profile for the fleet-status cell below).
sys.path.insert(0, str(NB_DIR))
import run_study


In [ ]:
# Importing rather than re-declaring is what stops this notebook and a
# headless fleet run from drifting apart. A hand-copied roster or sweep
# config here -- MODELS, COT_ARGS, or the config build_config assembles --
# would be exactly how two nominally "identical" runs stop being identical:
# an edit to run_study.py (the file the fleet actually executes) would
# otherwise leave this notebook silently validating a stale config.
from run_study import MODELS, COT_ARGS, build_config, selected_model

# selected_model is imported for parity with the per-lane driver's public
# API, not called here: it resolves and validates a lane's own LEAN_MODEL
# environment variable, and this notebook has no lane of its own to
# validate -- it checks prompt CONTENTS once, against one real theorem, not
# any particular checkpoint's serving lane.
_sample_key = sorted(MODELS)[0]
_sample_config = build_config(_sample_key)
print(
    f"{len(MODELS)} models x rungs {_sample_config['rungs']} x "
    f"{_sample_config['n_replicates']} replicate(s), "
    f"{_sample_config['theorems']['limit']} theorems"
)


## Prompt Validation


In [ ]:
# The replay_passing_* sidecars this reads were archived to S3 on 2026-08-25
# (notebooks/README.md); restore them under notebooks/deduction/data/ or point
# SMOLBENCH_LEAN_DATA at a tree that has them before running this cell.
from smolbench.deduction.lean import corpus, context

# Real corpus, not a fixture: replay_passing/novel_premises/val is exactly
# build_config's theorem source (see run_study.build_config, "theorems"),
# so this cell validates prompts the same way a live sweep would build
# them.
POOL = list(corpus.iter_replay_passing("novel_premises", "val"))

# k.strategy == "last" (build_config's fixed k selector): the config always
# asks for the FINAL proof step, so k here must match that, not an
# arbitrary in-range value.
SCAN_CAP = 60  # 39 of the first 60 real theorems exercise the padding path
               # below (verified); capping the scan keeps this cell fast
               # without walking the whole 805-theorem pool for a common
               # case.

chosen = None
for theorem in POOL[:SCAN_CAP]:
    if not theorem.traced_tactics:
        continue  # defensive: iter_replay_passing does not itself filter on has_proof
    k = len(theorem.traced_tactics) - 1
    hint2_tokens = context._count_tokens(context.render(theorem, k, "hint", 2).text)
    hint3_tokens = context._count_tokens(context.render(theorem, k, "hint", 3).text)
    # Selecting on this STRICT inequality is what makes the assertion below
    # non-vacuous: when hint:3 adds nothing over hint:2,
    # _render_noise_parts short-circuits and returns the hint:2 baseline
    # UNPADDED (see context._render_noise_parts's "Already exact" branch),
    # so "noise:3's token count matches hint:3's" would be trivially true
    # for a reason that has nothing to do with the padding logic actually
    # being exercised.
    if hint3_tokens > hint2_tokens:
        chosen = (theorem, k, hint2_tokens, hint3_tokens)
        break

if chosen is None:
    raise RuntimeError(
        f"scanned the first {SCAN_CAP} theorems of the replay_passing/"
        "novel_premises/val pool and found none where hint:3 adds tokens "
        "over hint:2 -- cannot validate the noise:3 length control against "
        "a non-trivial padding case. Raise SCAN_CAP or investigate whether "
        "the corpus/premise data has changed."
    )
theorem, k, hint2_tokens, hint3_tokens = chosen

# stepk:1 -- the plainest rung this study runs (full tactic state, no
# answer-conditional content).
print(context.render(theorem, k, "stepk", 1).text)

# noise:3 is the token-matched length control for hint:3: same hint:2
# baseline, padded with whitespace to hint:3's exact token count, so an
# accuracy gap between hint:3 and noise:3 can only be explained by content,
# never by prompt length.
noise3_tokens = context._count_tokens(context.render(theorem, k, "noise", 3).text)
assert noise3_tokens == hint3_tokens, (
    f"{theorem.full_name} k={k}: noise:3 is {noise3_tokens} tokens but "
    f"hint:3 is {hint3_tokens} tokens -- the length control is not binding"
)
assert hint3_tokens > hint2_tokens, (
    f"{theorem.full_name} k={k}: hint:3 ({hint3_tokens} tokens) does not "
    f"exceed hint:2 ({hint2_tokens} tokens) -- this theorem should never "
    "have been selected"
)
print(
    f"{theorem.full_name} k={k}: hint:2={hint2_tokens} tokens, "
    f"hint:3={hint3_tokens} tokens, noise:3={noise3_tokens} tokens "
    "(noise:3 matches hint:3 exactly, over a real content gap)"
)

# Pure CPU: context._count_tokens is a self-contained ad hoc counter (see
# its docstring) -- this cell downloads no tokenizer, and touches no
# network and no AWS.


## Fleet Launch

Like the induction notebook, this study does **not** run models from
cells, and there are no per-model cells here either: **running this
notebook cannot launch boxes.** The fleet supervisor,
`scripts/run_fleet.py`, is launched from a **terminal**, at the repo root:

```
nohup .venv/bin/python scripts/run_fleet.py --phase deduction > fleet.out 2>&1 &
```

Useful flags (see `scripts/run_fleet.py`'s own module docstring and
`--help` for the full set):

- `--phase {induction,deduction,both}` -- which subprocess phase(s) each
  lane runs this invocation (default: `induction`).
- `--lanes <comma-separated spec keys>` -- restrict to a subset of the 21
  lanes instead of the full roster (default: all of them).
- `--dry-run` -- print the launch plan (tier, command, full per-lane
  environment) and exit; launches no subprocess and makes no AWS call.
  **Run this first.** It proves the lane wiring is correct, not that the
  lanes will actually start -- it deliberately skips `preflight` (the
  per-lane tokenizer warm-up and completion-budget derivation) and the
  nightly-image digest lookup, both of which only run on a live launch.

### The reattach contract

**Config epoch.** Since 2026-08-18 every `EC2_DEPLOY_SPECS` entry serves the certified determinism bundle (prefix caching off, `--max-num-seqs 1`, `--enforce-eager`, `--seed 0`) plus digest/revision pins. Cross-config agreement was measured at 0/8 prompts, so anything this fleet collects now is config-incomparable with the 2026-08-16 study numbers -- a re-run is a new study, not more of the old one. (Record: `CONTAMINATION_INVENTORY_2026-08-15.md`, archived 2026-08-25 -- see `notebooks/README.md`.)

Under `--phase both`, the supervisor chains induction then deduction on
ONE lane and shuts the box down after the deduction phase's spool sync;
under `--phase deduction` alone, it REATTACHES each lane to the box its
own induction phase already provisioned, rather than provisioning a
second instance per model. The mechanism is `scripts/run_fleet.py`'s
`lane_env`: for `phase == "deduction"` it sets `LEAN_STATE_FILE` to the
SAME value as the induction phase's `INDUCTION_STATE_FILE`, and
`EC2_EXPERIMENT_TAG` is derived from the SAME `Lane.experiment_tag`
(`f"scaling-{key}"`) regardless of phase -- so `ec2.provision_spot_
instance()` (called with no arguments by `notebooks/deduction/
run_study.py`'s `main`) finds and reuses the live instance already
recorded under that tag/state file instead of launching a fresh one.
`--phase induction` alone never shuts a box down on its own -- see
"Teardown" below.

Lane logs land under `notebooks/induction/results/fleet_logs/<spec-key>.log`
-- the same per-lane log file both phases append to.


In [ ]:
# scripts/ has no __init__.py -- it imports as a namespace package once the
# repo root is on sys.path, which REPO_ROOT (derived above, in the keys.env
# cell) provides.
sys.path.insert(0, str(REPO_ROOT))          # REPO_ROOT is derived in the keys.env cell above
from scripts.fleet_status import fleet_rows, format_fleet_table

# READ-ONLY: fleet_rows() is a describe_instances call, server-side filtered
# on tag:smolbench:experiment = scaling-*, so it costs nothing and cannot
# start or stop anything. Needs AWS credentials; returns an explicit
# "no instances" line rather than blank output when the fleet is down, so a
# torn-down fleet is distinguishable from a credentials problem at a glance.
# The default tag prefix already covers deduction lanes: a lane keeps ONE
# smolbench:experiment tag (f"scaling-{key}") across both its induction and
# deduction phases (see run_fleet.Lane.experiment_tag), so this same query
# lists deduction-phase instances with no changes needed.
print(format_fleet_table(fleet_rows()))


## Analysis

**Results analysis runs on remote compute, not this host** (same policy as
the induction notebook). The cells below are written out so the analysis
is reproducible and reviewable from this notebook, but they are gated
behind an explicit opt-in so that a casual run-all on a laptop does not
pull anything down or launch a Lean verification pass.

**Results are S3-only.** Nothing accumulates locally: `notebooks/deduction/
run_study.py`'s `spool_to_s3` uploads each lane's run directory to
`s3://smolbench-results-414266451290/deduction/runs/scaling_<spec-key>/`
after `runner.sweep` returns, verifies every upload against S3's own
`ContentLength`, and only then prunes the local run directory down to
`manifest.json` (kept so a later resume can recognise the run without
re-downloading the whole spool first just to check).

**The generation -> verification split.** Generation runs on the main
`.venv` with a `NullVerifier`, so every cell row lands with
`verdict == "unverified"` -- the main venv (Python 3.14) cannot import the
real Lean verifier at all, since that module requires `lean_dojo`, which
pins `python<3.13`. Real Lean verdicts only appear after a SEPARATE pass,
`scripts/lean_verify_rows.py`, runs under the dedicated `.venv-lean`
(Python 3.12) environment, replaying each candidate proof tail against a
real Dojo session and writing `verified_rows.jsonl` beside `all_rows.jsonl`
in S3 -- never modifying the original. **Analysing `all_rows.jsonl` before
that pass runs shows all-zero success rates that are an artifact of an
unrun verification pass, not a finding about any model.**

**Not every cell is measurable even after that pass.** A lane is 944 cells, not 4x300 -- `stepk:1` renders for all 300 theorems, `hint:2` for 222, `hint:3`/`noise:3` for 211. In the 2026-08-16 study 232 of the 944 could never be judged (byte-identical cell keys in every lane); the 2026-08-18 DojoInit recovery restored 121, leaving 111 (11.8%) unmeasurable and a per-lane denominator of 833. Unmeasurable cells are excluded, never scored 0, while the five model-dependent no-survivor cells ARE counted as failures -- see `notebooks/deduction/error_bars.py`'s module docstring (`--no-count-as-failure` restores the old drop rule). (Records: `DEDUCTION_COVERAGE_DIAGNOSIS_2026-08-16.md`, `DOJOINIT_RECOVERY_2026-08-18.md`, archived 2026-08-25.)


In [ ]:
import os

ANALYSIS_HOST = os.environ.get("ANALYSIS_HOST", "").strip() == "1"
if not ANALYSIS_HOST:
    print("ANALYSIS_HOST != 1 -- skipping. Set ANALYSIS_HOST=1 on the analysis host to run.")
else:
    # Unlike the induction notebook's ANALYSIS_HOST branch (which calls
    # EXPERIMENT.harness.sync_down() in-kernel), this driver has no
    # equivalent harness/experiment object to call: notebooks/deduction/
    # run_study.py is a per-lane subprocess driver, not something that owns
    # a results-store handle this notebook could reach into. So this branch
    # prints the commands an analysis host actually runs, in order, rather
    # than running anything expensive itself.
    print(
        "On the analysis host, run (in order):\n\n"
        "  1. Verification pass, once per finished lane (.venv-lean; turns\n"
        "     'unverified' rows into real Lean verdicts, writing\n"
        "     verified_rows.jsonl beside all_rows.jsonl in S3):\n"
        "       .venv-lean/bin/python scripts/lean_verify_rows.py --runs 'scaling_<key>*'\n"
        "     (preview first with --dry-run -- exempt from the .venv-lean/RAM/\n"
        "     lock requirements above, so it also runs from the main .venv;\n"
        "     see --help for the full flag set. It is a sibling, off-limits\n"
        "     file to this notebook.)\n\n"
        "  2. Power analysis / figures, after verification -- see the next\n"
        "     cell."
    )


In [ ]:
# Power analysis: reads verified rows back from S3 (after the verification
# NOTE: the study's published deduction numbers do NOT come from this script.
# They come from notebooks/deduction/error_bars.py (block sign-flip over theorem
# blocks = the primary test, BCa CIs at B=500,000; 14 of 21 ladder contrasts
# reject under Holm) and hint_vs_noise.py. power_analysis.py is replicate-design
# sizing only (record: FAMILY_LADDER_ANALYSIS_2026-08-16.md, archived 2026-08-25).
# pass above has run) and reports on this study's replicate design. Shown
# as a shell comment, not executed here -- it runs on the analysis host,
# not in this kernel.
#
# uv run --no-project --with numpy --with scipy --with statsmodels \
#     python notebooks/deduction/power_analysis.py --s3


## Teardown

No teardown cell here -- teardown is a fleet-lifecycle operation, not a
notebook one:

```
.venv/bin/python scripts/fleet_teardown.py                 # read-only listing
.venv/bin/python scripts/fleet_teardown.py --terminate     # actually terminate
```

A successful `--phase deduction` (or `--phase both`) run shuts each lane's
box down itself once the deduction phase's spool sync finishes (see
`scripts/run_fleet.py`'s module docstring, "Phases and the
reuse-then-shutdown lifecycle"), so `fleet_teardown.py` is the **safety
net** -- for lanes that stalled partway, an `--phase induction`-only run
(which never shuts a box down on its own), and anything the supervisor
otherwise lost track of.

**Warning:** `notebooks/deduction/run_study.py --teardown` exists but is
**standalone-only** -- its own `--teardown` flag's help text says so
explicitly: "STANDALONE USE ONLY: under the fleet, the supervisor owns
instance lifecycle and tears the box down itself once every phase
scheduled for this lane has finished -- do not pass this flag from
fleet-driven automation." It exists purely for a solo smoke test of that
file with nothing else depending on the box; passed against a fleet lane,
it would terminate the instance out from under the supervisor's own
end-of-lifecycle bookkeeping.
